# 04 - Shrinkage and Fixes

Replace the sample covariance with a **Ledoit-Wolf** shrinkage estimator and watch the plug-in weights stabilize.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from markowitz import LedoitWolfCovariance, MeanVariance

rng = np.random.default_rng(7)
n, T, n_draws = 20, 60, 200

mu_true = rng.uniform(0.04, 0.10, size=n)
A = rng.standard_normal((n, n))
Sigma_true = A @ A.T / n + 0.05 * np.eye(n)
L = np.linalg.cholesky(Sigma_true)


## Refit with shrunk covariance

In [ ]:
weights_sample = np.empty((n_draws, n))
weights_shrunk = np.empty((n_draws, n))
intensities = np.empty(n_draws)

for k in range(n_draws):
    R = mu_true + (rng.standard_normal((T, n)) @ L.T)
    mu_hat = R.mean(axis=0)
    Sigma_sample = np.cov(R, rowvar=False, ddof=1)
    lw = LedoitWolfCovariance().fit(R)
    intensities[k] = lw.shrinkage_

    weights_sample[k] = MeanVariance(risk_aversion=3.0).fit(mu_hat, Sigma_sample).weights_
    weights_shrunk[k] = MeanVariance(risk_aversion=3.0).fit(mu_hat, lw.covariance_).weights_


## Compare weight dispersion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].boxplot(weights_sample, showfliers=False)
axes[0].set_title('Plug-in: sample covariance')
axes[1].boxplot(weights_shrunk, showfliers=False)
axes[1].set_title('Plug-in: Ledoit-Wolf covariance')
for ax in axes:
    ax.axhline(0.0, color='black', lw=0.5)
    ax.set_xlabel('asset index')
axes[0].set_ylabel('weight across draws')
fig.tight_layout()


## Shrinkage intensities

Ledoit-Wolf chooses $\delta^\star$ adaptively per sample. With $T = 60$ observations of $n = 20$ assets, intensities cluster well above 0.5 - the sample covariance is a poor estimator here.

In [ ]:
print(f'mean shrinkage intensity: {intensities.mean():.3f}')
print(f'std  shrinkage intensity: {intensities.std():.3f}')
print(f'mean |w| sample:  {np.abs(weights_sample).mean():.3f}')
print(f'mean |w| shrunk:  {np.abs(weights_shrunk).mean():.3f}')


## Takeaway

Shrinkage targets the inverse covariance directly, suppressing the small-eigenvalue directions that drove the instability in notebook 03. Notebook 05 brings in a different fix: a Bayesian prior on $\mu$.